In [0]:
%run ./00_logging

In [0]:
%run ./01_config_schemas

DataFrame[]

In [0]:
# Databricks notebook source
# MAGIC %run ./00_logging

# MAGIC %run ./01_config_schemas

from datetime import datetime
import pyspark.sql.functions as F

# Get widgets/parameters
dbutils.widgets.text("batch_date", "2026-10-01")
dbutils.widgets.text("load_type", "full")

BATCH_DATE = dbutils.widgets.get("batch_date")
LOAD_TYPE = dbutils.widgets.get("load_type")

print(f"Running Silver Layer for batch_date={BATCH_DATE}, load_type={LOAD_TYPE}")

# Ensure quarantine table exists
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.silver.quarantine (
        entity STRING,
        batch_date STRING,
        load_type STRING,
        reason STRING,
        record STRING,
        load_timestamp TIMESTAMP
    ) USING DELTA
""")

def transform_commits(df):
    return (df
        .withColumn("author_email_hash", F.sha2(F.col("commit.author.email"), 256))
        .withColumn("committer_email_hash", F.sha2(F.col("commit.committer.email"), 256))
        .withColumn("author_login", F.col("author.login"))
        .withColumn("committer_login", F.col("committer.login"))
        .withColumn("commit_date", F.expr("try_to_timestamp(commit.author.date)"))
        .withColumn("commit_message", F.col("commit.message"))
        .select("repo_name", "sha", "commit_message", "author_login", "committer_login",
                "author_email_hash", "committer_email_hash", "commit_date", "batch_date", "load_type"))

def transform_issues(df):
    return (df
        .withColumn("user_login", F.col("user.login"))
        .withColumn("is_pr", F.col("pull_request").isNotNull())
        .withColumn("label_count", F.size("labels"))
        .withColumn("created_at_ts", F.expr("try_to_timestamp(created_at)"))
        .withColumn("closed_at_ts", F.expr("try_to_timestamp(closed_at)"))
        .withColumn("updated_at_ts", F.expr("try_to_timestamp(updated_at)"))
        .withColumn("time_to_close_hours", F.when(F.col("closed_at_ts").isNotNull(),
            (F.col("closed_at_ts").cast("long") - F.col("created_at_ts").cast("long")) / 3600.0))
        .select("repo_name", "id", "number", "title", "state", "is_pr", "user_login", "label_count",
                "comments", "created_at_ts", "closed_at_ts", "updated_at_ts",
                "time_to_close_hours", "batch_date", "load_type"))

def transform_pulls(df):
    return (df
        .withColumn("user_login", F.col("user.login"))
        .withColumn("created_at_ts", F.expr("try_to_timestamp(created_at)"))
        .withColumn("closed_at_ts", F.expr("try_to_timestamp(closed_at)"))
        .withColumn("merged_at_ts", F.expr("try_to_timestamp(merged_at)"))
        .withColumn("updated_at_ts", F.expr("try_to_timestamp(updated_at)"))
        .withColumn("time_to_merge_hours", F.when(F.col("merged_at_ts").isNotNull(),
            (F.col("merged_at_ts").cast("long") - F.col("created_at_ts").cast("long")) / 3600.0))
        .select("repo_name", "id", "number", "title", "state", "merged", "user_login",
                "comments", "review_comments", "commits", "additions", "deletions", "changed_files",
                "created_at_ts", "closed_at_ts", "merged_at_ts", "updated_at_ts",
                "time_to_merge_hours", "batch_date", "load_type"))

TRANSFORM_FUNCS = {
    "commits": transform_commits,
    "issues": transform_issues,
    "pulls": transform_pulls
}

TS_COL = {"commits": "commit_date", "issues": "created_at_ts", "pulls": "created_at_ts"}

def process_silver_entity(entity):
    start = datetime.now()
    param = f"{LOAD_TYPE}/{BATCH_DATE}/{entity}"
    src, target = f"workspace.bronze.{entity}", f"workspace.silver.{entity}"
    try:
        df_b = spark.table(src).filter((F.col("batch_date") == BATCH_DATE) & (F.col("load_type") == LOAD_TYPE))
        if df_b.limit(1).count() == 0:
            write_log("bronze_to_silver", entity, param, LOAD_TYPE, start, "SUCCESS", 0, 0, "no rows in bronze for batch")
            print(f"Silver {entity}: no rows for this batch")
            return

        df = TRANSFORM_FUNCS[entity](df_b).withColumn("load_timestamp", F.current_timestamp())
        ok = " AND ".join(f"{k} IS NOT NULL" for k in KEYS[entity]) + f" AND {TS_COL[entity]} IS NOT NULL"

        bad = df.filter(f"NOT ({ok})")
        n_bad = bad.count()
        if n_bad:
            (bad.select(F.lit(entity).alias("entity"), "batch_date", "load_type",
                        F.lit("null key or unparseable date").alias("reason"),
                        F.to_json(F.struct("*")).alias("record"), "load_timestamp")
                .write.mode("append").option("mergeSchema", "true").saveAsTable("workspace.silver.quarantine"))

        good = df.filter(ok).dropDuplicates(KEYS[entity])
        if not spark.catalog.tableExists(target):
            good.limit(0).write.format("delta").saveAsTable(target)
        good.createOrReplaceTempView("silver_src")

        cond = " AND ".join(f"t.{k} = s.{k}" for k in KEYS[entity])
        matched = "" if entity == "commits" else "AND s.updated_at_ts >= t.updated_at_ts"
        res = spark.sql(f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} t USING silver_src s ON {cond}
            WHEN MATCHED {matched} THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *""").collect()[0]
        ins, upd = int(res["num_inserted_rows"]), int(res["num_updated_rows"])
        write_log("bronze_to_silver", entity, param, LOAD_TYPE, start, "SUCCESS", ins, upd, f"quarantined_rows={n_bad}")
        print(f"Silver {entity}: inserted={ins} updated={upd} quarantined={n_bad}")
    except Exception as e:
        write_log("bronze_to_silver", entity, param, LOAD_TYPE, start, "FAILURE", 0, 0, str(e)[:500])
        print(f"Silver {entity}: FAILED -> {str(e)[:300]}")

for e in ["commits", "issues", "pulls"]:
    process_silver_entity(e)

Running Silver Layer for batch_date=2026-10-10, load_type=incremental
Silver commits: inserted=0 updated=87 quarantined=0
Silver issues: inserted=0 updated=289 quarantined=0
Silver pulls: inserted=0 updated=235 quarantined=0


In [0]:
%sql
SELECT 
    layer, 
    entity, 
    load_type, 
    status, 
    rows_inserted, 
    rows_updated, 
    start_time,
    end_time
FROM workspace.ops.pipeline_execution_logs 
ORDER BY start_time DESC;

layer,entity,load_type,status,rows_inserted,rows_updated,start_time,end_time
bronze_to_silver,pulls,incremental,SUCCESS,0,235,2026-10-10T09:27:58.768Z,2026-10-10T09:28:06.318Z
bronze_to_silver,issues,incremental,SUCCESS,0,289,2026-10-10T09:27:49.723Z,2026-10-10T09:27:56.851Z
bronze_to_silver,commits,incremental,SUCCESS,0,87,2026-10-10T09:27:40.012Z,2026-10-10T09:27:48.024Z
bronze_to_silver,pulls,incremental,SUCCESS,235,0,2026-10-10T09:26:52.893Z,2026-10-10T09:26:59.496Z
bronze_to_silver,issues,incremental,SUCCESS,289,0,2026-10-10T09:26:44.935Z,2026-10-10T09:26:51.258Z
bronze_to_silver,commits,incremental,SUCCESS,0,87,2026-10-10T09:26:36.533Z,2026-10-10T09:26:43.193Z
bronze_to_silver,commits,full,SUCCESS,0,0,2026-10-10T09:26:09.418Z,2026-10-10T09:26:09.871Z
bronze_to_silver,pulls,full,SUCCESS,0,0,2026-10-10T09:26:02.913Z,2026-10-10T09:26:03.743Z
bronze_to_silver,issues,full,SUCCESS,0,0,2026-10-10T09:26:00.257Z,2026-10-10T09:26:01.091Z
bronze_to_silver,commits,full,SUCCESS,0,0,2026-10-10T09:25:57.134Z,2026-10-10T09:25:57.964Z
